# AgenticML — Experimento 1: Iris (Classificação Multiclasse)

**Dataset:** Iris (UCI / sklearn)  
**Tipo de problema:** Classificação multiclasse (3 classes)  
**Objetivo:** Classificar a espécie da flor de íris com base em medidas das pétalas e sépalas  

---
**Contexto:** Este notebook documenta a execução completa do pipeline AgenticML no dataset Iris,
registrando as decisões de cada agente, o modelo selecionado e as métricas obtidas para comparação
com o baseline Auto-sklearn.

In [1]:
import sys, os, time, warnings
warnings.filterwarnings('ignore')

# Adiciona raiz do projeto ao path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
from sklearn.datasets import load_iris

from core.graph import build_graph
from core.state import initial_state
from tcc_eval import make_split, evaluate_agenticml


## 1. Dataset — Iris

| Atributo | Valor |
|----------|-------|
| Instâncias | 150 |
| Features | 4 (sepal_length, sepal_width, petal_length, petal_width) |
| Classes | 3 (setosa, versicolor, virginica) — balanceadas (50 cada) |
| Missing values | Nenhum |
| Fonte | `sklearn.datasets.load_iris()` |

In [2]:
iris = load_iris(as_frame=True)
df = iris.frame.copy()
df.columns = [c.replace(' (cm)', '').replace(' ', '_') for c in df.columns]
df['species'] = iris.target_names[df['target']]
df = df.drop(columns=['target'])

print('Shape:', df.shape)
print('\nDistribuição da target:')
print(df['species'].value_counts())
display(df.head())

Shape: (150, 5)

Distribuição da target:
species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [3]:
# Converte para CSV (entrada da API)
csv_bytes = df.to_csv(index=False).encode('utf-8')
print(f'CSV gerado: {len(csv_bytes):,} bytes')

CSV gerado: 3,858 bytes


## 2. Execução do Pipeline AgenticML

In [4]:
PROMPT = (
    "Classificar a espécie da flor de íris (setosa, versicolor ou virginica) "
    "com base nas medidas das pétalas e sépalas."
)

# Protocolo de avaliação (parecer da orientação): holdout isolado da seleção de modelos.
# O AgenticML treina e faz validação cruzada só no train_df; o test_df nunca é visto
# durante a seleção — a métrica reportada como final vem do holdout (ver Agente 5 abaixo).
SEED = 42
train_df, test_df, split_info = make_split(
    df, target='species', task='classification', test_size=0.2, seed=SEED
)
print(f"Split: {split_info['n_train']} treino / {split_info['n_test']} teste "
      f"(estratificado={split_info['stratified']}, seed={SEED})")

t0 = time.time()
ev = evaluate_agenticml(
    train_df, test_df, target='species', prompt=PROMPT,
    hint='classification', seed=SEED, return_state=True,
)
elapsed = time.time() - t0
result = ev['_final_state']

print(f'\n✓ Pipeline concluído em {elapsed:.2f}s')
print('Status final:', result.get('status'))


Split: 120 treino / 30 teste (estratificado=True, seed=42)

✓ Pipeline concluído em 18.40s
Status final: done


In [5]:
# Diagnóstico rápido — execute esta célula antes de inspecionar os resultados por agente
status = result.get('status')
output = result.get('output') or {}
print(f'Status do pipeline: {status}')
if status == 'done' and output.get('status') == 'error':
    # Short-circuit: Agente 6 rodou em modo erro
    print(f"Falhou em: {output.get('failed_at')}")
    print(f"Erro técnico: {(output.get('explanation') or {}).get('technical', '')}")
elif result.get('failed_at'):
    print(f"Falhou em: {result.get('failed_at')}")
    print(f"Erro: {(result.get('error') or {}).get('technical', '')}")
else:
    print('✓ Pipeline concluído com sucesso — todos os agentes executaram.')


Status do pipeline: done
✓ Pipeline concluído com sucesso — todos os agentes executaram.


## 3. Resultados por Agente

### Agente 1 — Ingestão e Inspeção

In [6]:
inspection = result.get('inspection') or {}
schema = inspection.get('schema', {})

print('=== Schema ===')
print('  Colunas:', schema.get('columns'))
print('  Shape:  ', schema.get('shape'))
print('  Tipos:  ', schema.get('dtypes'))
print()
print('Encoding detectado:', inspection.get('encoding_detected'))
print('Separador detectado:', repr(inspection.get('separator_detected')))

print('\n=== Estatísticas Descritivas ===')
stats = inspection.get('descriptive_stats', {})
display(pd.DataFrame(stats).T)

=== Schema ===
  Colunas: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']
  Shape:   [120, 5]
  Tipos:   {'sepal_length': 'float64', 'sepal_width': 'float64', 'petal_length': 'float64', 'petal_width': 'float64', 'species': 'str'}

Encoding detectado: utf-8
Separador detectado: ','

=== Estatísticas Descritivas ===


,nulls,mean,std,min,max,median,unique_count,top_values
sepal_length,0.0,5.841667,0.840926,4.3,7.9,5.75,NaN,NaN
sepal_width,0.0,3.048333,0.448524,2.0,4.4,3.0,NaN,NaN
petal_length,0.0,3.77,1.76852,1.1,6.9,4.25,NaN,NaN
petal_width,0.0,1.205,0.762663,0.1,2.5,1.3,NaN,NaN
species,0,NaN,NaN,NaN,NaN,NaN,3,"{'setosa': 40, 'virginica': 40, 'versicolor': 40}"


### Agente 2 — Diagnóstico de Qualidade

In [7]:
quality = result.get('quality') or {}
issues = quality.get('issues', [])

print(f"Overall severity: {quality.get('overall_severity')}")
print(f"Issues detectados: {len(issues)}")

if issues:
    for i in issues:
        print(f"  [{i['severity'].upper()}] {i['type']}: {i['alert']}")
else:
    print('  ✓ Nenhum problema de qualidade detectado.')

Overall severity: medium
Issues detectados: 2
  [MEDIUM] high_correlation: As colunas 'petal_length' e 'petal_width' apresentam uma alta correlação de 0.9637. Isso pode indicar que uma das colunas é redundante para o modelo, podendo ser removida ou utilizada com cautela para evitar problemas de multicolinearidade.
  [LOW] duplicate_rows: Foi detectada a presença de 1 linha duplicada, o que representa 0.83% do total de dados. Recomenda-se remover as linhas duplicadas para evitar que influenciem desproporcionalmente o modelo.


### Agente 3 — Feature Engineering

In [8]:
features = result.get('features') or {}
transforms = features.get('transformations_applied', [])

print(f"Coluna target: {features.get('target_column')}")
print(f"Features pós-transformação ({len(features.get('feature_columns', []))}): {features.get('feature_columns')}")
print()
print(f"Transformações aplicadas ({len(transforms)}):")
for t in transforms:
    print(f"  • {t['type']} em {t['columns']}")
    print(f"    → {t['justification']}")

Coluna target: species
Features pós-transformação (4): ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

Transformações aplicadas (1):
  • standard_scaler em ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
    → A padronização é adequada para as medidas das sépalas e pétalas, pois não há limites naturais e queremos garantir que todas as features numéricas tenham a mesma escala para melhorar o desempenho do modelo.


### Agente 4 — Interpretação de Linguagem Natural

In [9]:
interp = result.get('interpretation') or {}

print(f"Tipo de problema: {interp.get('problem_type')}")
print(f"Confiança:        {interp.get('confidence')}")
print(f"Divergiu do hint: {interp.get('diverged_from_user')}")
print(f"\nJustificativa:\n  {interp.get('justification')}")

Tipo de problema: classification
Confiança:        high
Divergiu do hint: False

Justificativa:
  O objetivo do usuário é classificar a espécie da flor de íris (setosa, versicolor ou virginica) com base nas medidas das pétalas e sépalas. A coluna target identificada é 'species', que é uma variável categórica com três possíveis classes. As colunas de features são medidas numéricas das pétalas e sépalas, que serão usadas para prever a categoria da espécie. Portanto, este é claramente um problema de classificação, pois envolve prever uma categoria discreta com base em variáveis de entrada contínuas.


### Agente 5 — Seleção e Treinamento

In [10]:
training = result.get('training') or {}

print(f"Modelo selecionado: {training.get('model_selected')}")
print(f"Métrica utilizada:  {training.get('metric_used')}")
print(f"Valor da métrica (CV, seleção): {training.get('metric_value')}")
print()
print('Todos os candidatos:')
candidates = training.get('candidate_results', [])
df_cands = pd.DataFrame(candidates).sort_values('score', ascending=False)
display(df_cands)
print(f"\nJustificativa:\n  {training.get('justification')}")

print('\n=== Validação honesta (holdout isolado, nunca visto na seleção) ===')
print(f"Métrica no holdout: {ev['holdout_metric_name']}={ev['holdout_metric']}")
print(f"Gap de otimismo (CV − holdout): {ev['optimism_gap']}")
if ev['optimism_gap'] is not None and abs(ev['optimism_gap']) > 0.02:
    print('  ⚠ Gap > 0.02 — reportar o valor de holdout no TCC, não o de CV.')


Modelo selecionado: SVM
Métrica utilizada:  f1_weighted
Valor da métrica (CV, seleção): 0.9665

Todos os candidatos:


,model,score
3,SVM,0.9665
0,Logistic Regression,0.9663
1,Random Forest,0.9580
2,XGBoost,0.9580



Justificativa:
  SVM foi selecionado com F1-score ponderado de 0.9665 para o problema de classification. Demais candidatos avaliados: Logistic Regression (0.9663), XGBoost (0.9580), Random Forest (0.9580). Hiperparâmetros otimizados via busca aleatória (CV): {'C': 31.4288, 'gamma': 0.0043}.

=== Validação honesta (holdout isolado, nunca visto na seleção) ===
Métrica no holdout: f1_weighted=0.9666
Gap de otimismo (CV − holdout): -0.0001


### Agente 6 — Previsão e Explicação

In [11]:
output = result.get('output') or {}
explanation = output.get('explanation', {})
importances = output.get('feature_importances', [])

print('=== Explicação Técnica ===')
print(explanation.get('technical', ''))

print('\n=== Explicação Acessível ===')
print(explanation.get('accessible', ''))

if importances:
    print('\n=== Importância das Features ===')
    display(pd.DataFrame(importances))

=== Explicação Técnica ===
O pipeline AgenticML foi aplicado a um conjunto de dados de 120 linhas e 5 colunas, com o objetivo de classificar a espécie de flores de íris. O Agente 2 identificou problemas de qualidade, como alta correlação entre algumas features e linhas duplicadas, com severidade geral média. No entanto, essas questões não foram consideradas críticas para o processo de modelagem. O Agente 3 aplicou a transformação de dados usando o `standard_scaler` nas colunas de medidas das pétalas e sépalas, preparando os dados para o modelo de classificação.

O Agente 4 confirmou que o problema é de classificação, pois a variável alvo 'species' é categórica com três classes distintas. O Agente 5 conduziu o treinamento de modelos, selecionando o SVM como o melhor candidato com um F1-score ponderado de 0.9665, ligeiramente superior à Regressão Logística (0.9663) e significativamente melhor que Random Forest e XGBoost (ambos com 0.9580). A escolha do SVM foi baseada na sua performance 

## 4. Resumo

In [12]:
summary = {
    'Experimento':             'Iris — Classificação',
    'Dataset':                 f"{schema.get('shape', [0,0])[0]} linhas × {schema.get('shape', [0,0])[1]} colunas",
    'Split':                   f"{split_info['n_train']} treino / {split_info['n_test']} teste (seed={SEED})",
    'Tipo de problema':        interp.get('problem_type', 'N/A'),
    'Issues de qualidade':     len(issues),
    'Transformações':          len(transforms),
    'Modelo selecionado':      training.get('model_selected', 'N/A'),
    'Métrica':                 training.get('metric_used', 'N/A'),
    'Score CV (seleção)':      training.get('metric_value', 'N/A'),
    'Score holdout (honesto)': ev['holdout_metric'],
    'Gap de otimismo':         ev['optimism_gap'],
    'Tempo (s)':               round(elapsed, 2),
}

display(pd.DataFrame([summary]).T.rename(columns={0: 'Valor'}))


,Valor
Experimento,Iris — Classificação
Dataset,120 linhas × 5 colunas
Split,120 treino / 30 teste (seed=42)
Tipo de problema,classification
Issues de qualidade,2
Transformações,1
Modelo selecionado,SVM
Métrica,f1_weighted
Score CV (seleção),0.9665
Score holdout (honesto),0.9666
